# Обучение Feedforward Neural Network

In [1]:
import random

import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

SEED = 52

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)


In [2]:
X_train = pd.read_csv('data/X_train_encoded.csv')
X_test = pd.read_csv('data/X_test_encoded.csv')
y_train = pd.read_csv('data/y_train.csv')
y_test = pd.read_csv('data/y_test.csv')

In [3]:
X_train.head(3)

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,relationship_Own-child,relationship_Unmarried,relationship_Wife,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Male,is_usa_no,is_usa_yes
0,59,14,0,0,40,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,1
1,23,9,0,0,40,0,0,1,0,0,...,1,0,0,0,0,0,1,1,0,1
2,57,10,7688,0,45,0,0,1,0,0,...,0,0,0,0,0,0,1,1,0,1


In [4]:
DEVICE = 'cpu'
BATCH_SIZE = 100

## Гипотеза 1
Простая модель с одним скрытым слоем размерности 32

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self, in_features, hidden_size):
        super().__init__()
        self.l1 = nn.Linear(in_features, hidden_size)
        self.act = nn.ReLU()
        self.outl = nn.Linear(hidden_size, 1)
        self.final_act = nn.Sigmoid()

    def forward(self, x):
        y = self.l1(x)
        y = self.act(y)
        y = self.outl(y)
        y = self.final_act(y)
        return y

In [6]:
model = NeuralNetwork(98, 32)

In [7]:
model.to(DEVICE)

NeuralNetwork(
  (l1): Linear(in_features=98, out_features=32, bias=True)
  (act): ReLU()
  (outl): Linear(in_features=32, out_features=1, bias=True)
  (final_act): Sigmoid()
)

Выделяем валидационную выборку из train

In [8]:
X_train, X_val, y_train , y_val = train_test_split(X_train, y_train, shuffle=True,
                                                    random_state=52, test_size=0.2)

Переносим данные на наш `DEVICE`

In [9]:
X_val.head(3)

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,relationship_Own-child,relationship_Unmarried,relationship_Wife,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Male,is_usa_no,is_usa_yes
6585,24,13,0,0,40,0,0,1,0,0,...,1,0,0,0,0,0,1,0,0,1
17886,23,10,0,0,25,0,0,1,0,0,...,0,0,0,1,0,0,0,1,1,0
24810,20,10,0,0,17,0,0,1,0,0,...,1,0,0,0,0,0,1,0,0,1


In [10]:
y_train

,income
11587,1
15050,0
18563,0
22705,0
24009,0
...,...
4247,0
24077,0
11019,0
9884,0


In [11]:
X_train_pt = torch.tensor(X_train.to_numpy()).to(torch.float32).to(DEVICE)
X_test_pt = torch.tensor(X_test.to_numpy()).to(torch.float32).to(DEVICE)
X_val_pt = torch.tensor(X_val.to_numpy()).to(torch.float32).to(DEVICE)
y_train_pt = torch.tensor(y_train.to_numpy()).to(torch.float32).to(DEVICE)
y_test_pt = torch.tensor(y_test.to_numpy()).to(torch.float32).to(DEVICE)
y_val_pt = torch.tensor(y_val.to_numpy()).to(torch.float32).to(DEVICE)

(X_train_pt.shape, X_test_pt.shape, X_val_pt.shape,
 y_train_pt.shape, y_test_pt.shape, y_val_pt.shape)

(torch.Size([20838, 45]),
 torch.Size([6513, 45]),
 torch.Size([5210, 45]),
 torch.Size([20838, 1]),
 torch.Size([6513, 1]),
 torch.Size([5210, 1]))

Создаём DataLoaders, которые будут батчами выдавать данные для обучения нейросети

In [12]:
class AdultDataSet(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, index):
        return (self.X[index], self.y[index])

    def __len__(self):
        return len(self.X)

In [13]:
train_ds = AdultDataSet(X_train_pt, y_train_pt)
test_ds = AdultDataSet(X_test_pt, y_test_pt)
val_ds = AdultDataSet(X_val_pt, y_val_pt)

In [14]:
def make_loader(ds, shuffle=True, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, generator=g)

In [15]:
dl_train = make_loader(train_ds)
dl_val   = make_loader(val_ds)
dl_test  = make_loader(test_ds)

In [16]:
EPOCH_COUNT = 100

Инициализация модели

In [17]:
model = NeuralNetwork(X_train_pt.shape[1], 64).to(DEVICE)
loss_func = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Обучение

In [18]:
best_val_f1 = -1

for epoch in range(1, EPOCH_COUNT+1):
    model.train()

    for indx, (batch_x, batch_y) in enumerate(iter(dl_train), start=1):
        y_pred = model(batch_x) # [100, 1]
        loss = loss_func(y_pred, batch_y)  # порядок критичен!!!
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if indx % 50 == 0:
            y = batch_y.detach().cpu().numpy().astype(np.int64)
            y_pred = y_pred.detach().cpu().numpy()
            y_pred = np.where(y_pred > 0.5, 1, 0)
            f1_train = f1_score(y, y_pred, average='macro')
            print(f'[Train] F1: {f1_train:.4f}')

    model.eval()
    with torch.no_grad():
        f1s = []
        for batch_x, batch_y in dl_val:
            y_pred = model(batch_x)
            y = batch_y.cpu().numpy().astype(np.int64)
            y_pred = y_pred.cpu().numpy()
            y_pred = np.where(y_pred > 0.5, 1, 0)
            f1_val = f1_score(y, y_pred, average='macro')
            f1s.append(f1_val)
        f1_val_mean = torch.tensor(f1s).mean()

        if f1_val_mean > best_val_f1:
            best_val_f1 = f1_val_mean
            torch.save(model.state_dict(), 'models/best_model_weights.pth')

        print('-' * 100)
        print(f'[Val epoch={epoch:2d}] F1: {f1_val_mean:.4f}')
        print('-' * 100)

[Train] F1: 0.2278
[Train] F1: 0.4887
[Train] F1: 0.6533
[Train] F1: 0.6562
----------------------------------------------------------------------------------------------------
[Val epoch= 1] F1: 0.6115
----------------------------------------------------------------------------------------------------
[Train] F1: 0.4902
[Train] F1: 0.6001
[Train] F1: 0.5992
[Train] F1: 0.5464
----------------------------------------------------------------------------------------------------
[Val epoch= 2] F1: 0.6437
----------------------------------------------------------------------------------------------------
[Train] F1: 0.7647
[Train] F1: 0.6686
[Train] F1: 0.6189
[Train] F1: 0.7093
----------------------------------------------------------------------------------------------------
[Val epoch= 3] F1: 0.6380
----------------------------------------------------------------------------------------------------
[Train] F1: 0.5827
[Train] F1: 0.5464
[Train] F1: 0.5556
[Train] F1: 0.6710
------------

сохраним модель, показавшую лучший результат на валидационной выборке

In [19]:
model.load_state_dict(
    torch.load('models/best_model_weights.pth', weights_only=True)
)

<All keys matched successfully>

Проверяем предсказания модели на тесте

In [20]:
# финальный тест

model.eval()

with torch.no_grad():
    ys = []
    preds = []
    for batch_x, batch_y in dl_test:
        y_pred = model(batch_x)
        y = batch_y.cpu().numpy().astype(np.int64)
        y_pred = y_pred.cpu().numpy()
        y_pred = np.where(y_pred > 0.5, 1, 0)
        ys.extend(y.tolist())
        preds.extend(y_pred.tolist())

    f1_nn1 = f1_score(ys, preds, average='macro')
    accuracy_nn1 = accuracy_score(ys, preds)
    precision_nn1 = precision_score(ys, preds)
    recall_nn1 = recall_score(ys, preds)

In [21]:
metrics_nn1 = pd.DataFrame(
    columns=['f1-macro', 'accuracy', 'precision', 'recall'],
    data=[[f1_nn1, accuracy_nn1, precision_nn1, recall_nn1]],
    index=['FFNN с одним скрытым слоем']
)

In [22]:
metrics_nn1

,f1-macro,accuracy,precision,recall
FFNN с одним скрытым слоем,0.79193,0.852602,0.68368,0.675514


## Гипотеза 2
1й скрытый слой будет фича-экстрактором размерности 256, 2й слой - экстрактор фичей более высокого уровня, 3й - так же выходной.

In [23]:
class NeuralNetwork(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.l1 = nn.Linear(in_features, 256)
        self.l2 = nn.Linear(256, 64)
        self.l3 = nn.Linear(64, 1)

        self.act = nn.ReLU()
        self.final_act = nn.Sigmoid()

    def forward(self, x):
        y = self.act(self.l1(x))
        y = self.act(self.l2(y))
        y = self.final_act(self.l3(y))
        return y

In [24]:
EPOCH_COUNT = 100

Инициализация модели

In [25]:
model = NeuralNetwork(X_train_pt.shape[1]).to(DEVICE)
loss_func = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Обучение

In [26]:
best_val_f1 = -1
best_epoch = -1

for epoch in range(1, EPOCH_COUNT+1):
    model.train()

    for indx, (batch_x, batch_y) in enumerate(iter(dl_train), start=1):
        y_pred = model(batch_x) # [100, 1]
        loss = loss_func(y_pred, batch_y)  # порядок критичен!!!
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        f1s = []
        for batch_x, batch_y in dl_val:
            y_pred = model(batch_x)
            y = batch_y.cpu().numpy().astype(np.int64)
            y_pred = y_pred.cpu().numpy()
            y_pred = np.where(y_pred > 0.5, 1, 0)
            f1_val = f1_score(y, y_pred, average='macro')
            f1s.append(f1_val)
        f1_val_mean = torch.tensor(f1s).mean()

        if f1_val_mean > best_val_f1:
            best_val_f1 = f1_val_mean
            best_epoch = epoch
            torch.save(model.state_dict(), 'models/best_model_weights.pth')

        print(f'[Val epoch={epoch:2d}] F1: {f1_val_mean:.4f}')

print(f'Best epoch = {best_epoch}')

[Val epoch= 1] F1: 0.6059
[Val epoch= 2] F1: 0.6278
[Val epoch= 3] F1: 0.6349
[Val epoch= 4] F1: 0.6407
[Val epoch= 5] F1: 0.6645
[Val epoch= 6] F1: 0.7176
[Val epoch= 7] F1: 0.7297
[Val epoch= 8] F1: 0.7176
[Val epoch= 9] F1: 0.7004
[Val epoch=10] F1: 0.7565
[Val epoch=11] F1: 0.7229
[Val epoch=12] F1: 0.7409
[Val epoch=13] F1: 0.7342
[Val epoch=14] F1: 0.7516
[Val epoch=15] F1: 0.7569
[Val epoch=16] F1: 0.7336
[Val epoch=17] F1: 0.7543
[Val epoch=18] F1: 0.7437
[Val epoch=19] F1: 0.7553
[Val epoch=20] F1: 0.7564
[Val epoch=21] F1: 0.7611
[Val epoch=22] F1: 0.7377
[Val epoch=23] F1: 0.7382
[Val epoch=24] F1: 0.7333
[Val epoch=25] F1: 0.7355
[Val epoch=26] F1: 0.7510
[Val epoch=27] F1: 0.7658
[Val epoch=28] F1: 0.7537
[Val epoch=29] F1: 0.7831
[Val epoch=30] F1: 0.7548
[Val epoch=31] F1: 0.7692
[Val epoch=32] F1: 0.7338
[Val epoch=33] F1: 0.7716
[Val epoch=34] F1: 0.7648
[Val epoch=35] F1: 0.7735
[Val epoch=36] F1: 0.7774
[Val epoch=37] F1: 0.7801
[Val epoch=38] F1: 0.7737
[Val epoch=3

Сохраняем модель с лучшим результатом на валидационной выборке

In [27]:
model.load_state_dict(
    torch.load('models/best_model_weights.pth', weights_only=True)
)

<All keys matched successfully>

Проверяем предсказания модели на тесте

In [28]:
# финальный тест

model.eval()

with torch.no_grad():
    ys = []
    preds = []
    for batch_x, batch_y in dl_test:
        y_pred = model(batch_x)
        y = batch_y.cpu().numpy().astype(np.int64)
        y_pred = y_pred.cpu().numpy()
        y_pred = np.where(y_pred > 0.5, 1, 0)
        ys.extend(y.tolist())
        preds.extend(y_pred.tolist())

    f1_nn2 = f1_score(ys, preds, average='macro')
    accuracy_nn2 = accuracy_score(ys, preds)
    precision_nn2 = precision_score(ys, preds)
    recall_nn2 = recall_score(ys, preds)

In [29]:
metrics_nn2 = pd.DataFrame(
    columns=['f1-macro', 'accuracy', 'precision', 'recall'],
    data=[[f1_nn2, accuracy_nn2, precision_nn2, recall_nn2]],
    index=['FFNN с двумя скрытыми слоями']
)

In [30]:
metrics_nn2

,f1-macro,accuracy,precision,recall
FFNN с двумя скрытыми слоями,0.795231,0.853524,0.680601,0.69144


In [31]:
metrics_all_neural_network = pd.concat([metrics_nn1, metrics_nn2])

In [32]:
metrics_all_neural_network

,f1-macro,accuracy,precision,recall
FFNN с одним скрытым слоем,0.791930,0.852602,0.683680,0.675514
FFNN с двумя скрытыми слоями,0.795231,0.853524,0.680601,0.691440
